# Training vs Generated Data Analysis

Compares aggregate statistics, distributions and temporal structure between:
- **Reference**: the first `TRAINING_DATASET_SIZE` CSVs from the training directory (sorted)
- **Generated**: all CSVs produced by the reverse-diffusion model

Both datasets are either log-prices or log-returns.
Reference paths may be full-length stock histories; generated paths are fixed-length windows.

In [196]:
# ── USER INPUTS ───────────────────────────────────────────────────────────────
REF_DIRECTORY         = "../data/replication_returns_other"          # training data folder
GEN_DIRECTORY         = "../data/generated/replication/ODE_NO_NO_REPL_RET_OTHER_UNCO_ep-500_sde-ve_noise-exponential_20260504_115525_N2000_seed50"
SEED                  = 50
TRAINING_SEED         = 50    # must match config["train"]["seed"] used during training
VAL_SPLIT_RATIO       = 0.05 # set to e.g. 0.1 if --val_split_ratio was used; else None
TRAINING_DATASET_SIZE = None   # number of sliding windows — mirrors --train_subset_size
                               # set to None to use all available (post-val) windows
SEQ_LEN               = 2048  # window length used during training (--seq_len)
STRIDE                = 400   # stride used during training (--stride)
# ─────────────────────────────────────────────────────────────────────────────

In [197]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats as scipy_stats
from statsmodels.tsa.stattools import acf as sm_acf

np.random.seed(SEED)
rng = np.random.default_rng(SEED)

print(f"REF_DIRECTORY         : {REF_DIRECTORY}")
print(f"GEN_DIRECTORY         : {GEN_DIRECTORY}")
print(f"SEED                  : {SEED}")
print(f"TRAINING_DATASET_SIZE : {TRAINING_DATASET_SIZE}")
print(f"SEQ_LEN               : {SEQ_LEN}")
print(f"STRIDE                : {STRIDE}")

REF_DIRECTORY         : ../data/replication_returns_other
GEN_DIRECTORY         : ../data/generated/replication/ODE_NO_NO_REPL_RET_OTHER_UNCO_ep-500_sde-ve_noise-exponential_20260504_115525_N2000_seed50
SEED                  : 50
TRAINING_DATASET_SIZE : None
SEQ_LEN               : 2048
STRIDE                : 400


## 1. Load data

In [198]:
# ── Reference data ────────────────────────────────────────────────────────────
# Mirrors csdi_train_modified.py exactly:
#   1. Load ALL sorted CSVs — no file cap
#   2. Extract sliding windows with same SEQ_LEN / STRIDE / drop_incomplete=True
#   3. Replicate torch.randperm(seed) + val split + train_subset_size
import torch

ref_csv_files = sorted(Path(REF_DIRECTORY).glob("*.csv"))
if len(ref_csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {REF_DIRECTORY}")

all_windows: list[np.ndarray] = []
n_skipped_short = 0
for fp in ref_csv_files:
    series = pd.read_csv(fp)["log_adj_close"].values.astype(np.float64)
    T = len(series)
    max_start = T - SEQ_LEN
    if max_start < 0:   # shorter than SEQ_LEN — skip (drop_incomplete=True)
        n_skipped_short += 1
        continue
    for s in range(0, max_start + 1, STRIDE):
        all_windows.append(series[s : s + SEQ_LEN])

if n_skipped_short:
    print(f"[WARNING] {n_skipped_short} file(s) shorter than SEQ_LEN={SEQ_LEN} were skipped.")
if len(all_windows) == 0:
    raise RuntimeError("No windows could be extracted. Check SEQ_LEN / STRIDE / REF_DIRECTORY.")

print(f"Total windows in full dataset : {len(all_windows)} "
      f"(from {len(ref_csv_files) - n_skipped_short} usable file(s), "
      f"SEQ_LEN={SEQ_LEN}, STRIDE={STRIDE})")

# ── Replicate training subset selection with torch RNG ────────────────────────
torch_rng  = torch.Generator().manual_seed(TRAINING_SEED)
all_indices = torch.randperm(len(all_windows), generator=torch_rng).tolist()

# Carve out val split first (mirrors val_split_ratio logic in training script)
if VAL_SPLIT_RATIO is not None:
    val_size   = max(1, int(len(all_windows) * VAL_SPLIT_RATIO))
    train_pool = all_indices[val_size:]
    print(f"Val split : {val_size} windows held out (VAL_SPLIT_RATIO={VAL_SPLIT_RATIO})")
else:
    train_pool = all_indices

# Then take TRAINING_DATASET_SIZE from the remaining pool
if TRAINING_DATASET_SIZE is not None and TRAINING_DATASET_SIZE < len(train_pool):
    train_indices = train_pool[:TRAINING_DATASET_SIZE]
    print(f"Reference : {len(train_indices)} windows "
          f"(TRAINING_DATASET_SIZE={TRAINING_DATASET_SIZE} of {len(train_pool)} in pool)")
else:
    train_indices = train_pool
    if TRAINING_DATASET_SIZE is not None:
        print(f"[WARNING] TRAINING_DATASET_SIZE={TRAINING_DATASET_SIZE} >= pool size "
              f"({len(train_pool)}); using all pool windows.")
    print(f"Reference : {len(train_indices)} windows (full training pool)")

ref_paths = [all_windows[i] for i in train_indices]

assert all(len(w) == SEQ_LEN for w in ref_paths), "Window length mismatch — check SEQ_LEN"
print(f"  All window lengths = {SEQ_LEN}  ✓")

Total windows in full dataset : 5568 (from 210 usable file(s), SEQ_LEN=2048, STRIDE=400)
Val split : 278 windows held out (VAL_SPLIT_RATIO=0.05)
Reference : 5290 windows (full training pool)
  All window lengths = 2048  ✓


In [199]:
# ── Generated data ────────────────────────────────────────────────────────────
# Load ALL CSV files in GEN_DIRECTORY (no size cap).
# Two formats are supported:
#   (a) Per-sample CSVs  — columns: date, log_adj_close  (one file per path)
#   (b) Wide-format CSV  — columns: sample_idx, start_date, end_date, step_000, step_001, ...
#       (produced by generate_samples.py as "generated_samples.csv")
gen_csv_files = sorted(Path(GEN_DIRECTORY).glob("*.csv"))

if len(gen_csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in: {GEN_DIRECTORY}")

gen_paths: list[np.ndarray] = []
skipped = 0
for fp in gen_csv_files:
    df = pd.read_csv(fp)
    if "log_adj_close" in df.columns:
        # Format (a): individual per-sample CSV
        gen_paths.append(df["log_adj_close"].values.astype(np.float64))
    else:
        # Format (b): wide-format — each row is one sample path
        step_cols = sorted(
            [c for c in df.columns if c.startswith("step_")],
            key=lambda c: int(c.split("_")[1])
        )
        print(step_cols[999:], f"\n first column: {step_cols[0]}, last column: {step_cols[-1]}")
        if step_cols:
            print(df.head())
            for _, row in df.iterrows():
                gen_paths.append(row[step_cols].values.astype(np.float64))
        else:
            skipped += 1

if skipped:
    print(f"[WARNING] Skipped {skipped} file(s) with unrecognised column layout.")
if len(gen_paths) == 0:
    raise RuntimeError("No valid generated paths could be loaded from GEN_DIRECTORY.")

gen_lens = [len(p) for p in gen_paths]
print(f"Generated: {len(gen_paths)} paths loaded  ({len(gen_csv_files)} CSV file(s) read)")
print(f"  Path lengths — min={min(gen_lens)}  max={max(gen_lens)}  "
      f"median={int(np.median(gen_lens))}")

print(gen_paths[0][:5], "\n", gen_paths[0][-5:])
print(gen_paths[-1][:5], "\n", gen_paths[-1][-5:])

['step_999', 'step_1000', 'step_1001', 'step_1002', 'step_1003', 'step_1004', 'step_1005', 'step_1006', 'step_1007', 'step_1008', 'step_1009', 'step_1010', 'step_1011', 'step_1012', 'step_1013', 'step_1014', 'step_1015', 'step_1016', 'step_1017', 'step_1018', 'step_1019', 'step_1020', 'step_1021', 'step_1022', 'step_1023', 'step_1024', 'step_1025', 'step_1026', 'step_1027', 'step_1028', 'step_1029', 'step_1030', 'step_1031', 'step_1032', 'step_1033', 'step_1034', 'step_1035', 'step_1036', 'step_1037', 'step_1038', 'step_1039', 'step_1040', 'step_1041', 'step_1042', 'step_1043', 'step_1044', 'step_1045', 'step_1046', 'step_1047', 'step_1048', 'step_1049', 'step_1050', 'step_1051', 'step_1052', 'step_1053', 'step_1054', 'step_1055', 'step_1056', 'step_1057', 'step_1058', 'step_1059', 'step_1060', 'step_1061', 'step_1062', 'step_1063', 'step_1064', 'step_1065', 'step_1066', 'step_1067', 'step_1068', 'step_1069', 'step_1070', 'step_1071', 'step_1072', 'step_1073', 'step_1074', 'step_1075',

## 2. Aggregate statistics

In [200]:
def compute_stats(paths: list[np.ndarray], label: str) -> dict:
    flat = np.concatenate(paths)
    incs = np.concatenate([np.diff(p) for p in paths if len(p) > 1])
    return {
        "label"            : label,
        "n_paths"          : len(paths),
        "n_values"         : len(flat),
        "mean"             : flat.mean(),
        "std"              : flat.std(),
        "min"              : flat.min(),
        "q01"              : np.quantile(flat, 0.01),
        "q05"              : np.quantile(flat, 0.05),
        "q25"              : np.quantile(flat, 0.25),
        "q50"              : np.quantile(flat, 0.50),
        "q75"              : np.quantile(flat, 0.75),
        "q95"              : np.quantile(flat, 0.95),
        "q99"              : np.quantile(flat, 0.99),
        "max"              : flat.max(),
        "skewness"         : scipy_stats.skew(flat),
        "excess_kurtosis"  : scipy_stats.kurtosis(flat, fisher=True),
        "inc_mean"         : incs.mean(),
        "inc_std"          : incs.std(),
        "inc_min"          : incs.min(),
        "inc_q01"          : np.quantile(incs, 0.01),
        "inc_q99"          : np.quantile(incs, 0.99),
        "inc_max"          : incs.max(),
        "inc_skewness"     : scipy_stats.skew(incs),
        "inc_excess_kurt"  : scipy_stats.kurtosis(incs, fisher=True),
    }

ref_stats = compute_stats(ref_paths, "Reference (training)")
gen_stats = compute_stats(gen_paths, "Generated")

# ── Levels table ──────────────────────────────────────────────────────────────
level_keys = ["n_paths", "n_values", "mean", "std", "min", "q01", "q05",
              "q25", "q50", "q75", "q95", "q99", "max", "skewness", "excess_kurtosis"]
df_levels = pd.DataFrame(
    {k: {"Reference": ref_stats[k], "Generated": gen_stats[k]} for k in level_keys}
).T.round(4)

print("=" * 60)
print("  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)")
print("=" * 60)
display(df_levels.style.format("{:.4f}"))

  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)


,Reference,Generated
n_paths,5290.0000,1024.0000
n_values,10833920.0000,2097152.0000
mean,0.0005,0.0004
std,0.0208,0.0186
min,-0.9363,-0.8925
q01,-0.0562,-0.0463
q05,-0.0291,-0.0283
q25,-0.0086,-0.0105
q50,0.0000,0.0004
q75,0.0095,0.0111


In [201]:
for paths, label in [(ref_paths, "Reference"), (gen_paths, "GenWerated")]:
    flat = np.concatenate(paths)
    zero_pct = 100.0 * np.mean(flat == 0.0)
    print(f"{label}: {zero_pct:.4f}% absolute zeros ({(flat == 0.0).sum()} / {len(flat)})")


Reference: 7.1735% absolute zeros (777169 / 10833920)
GenWerated: 0.0000% absolute zeros (0 / 2097152)


In [202]:
# ── Increments table ──────────────────────────────────────────────────────────
inc_keys = ["inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99",
            "inc_max", "inc_skewness", "inc_excess_kurt"]
rename = {
    "inc_mean": "mean", "inc_std": "std", "inc_min": "min",
    "inc_q01": "q01", "inc_q99": "q99", "inc_max": "max",
    "inc_skewness": "skewness", "inc_excess_kurt": "excess_kurtosis",
}
df_incs = pd.DataFrame(
    {rename[k]: {"Reference": ref_stats[k], "Generated": gen_stats[k]}
     for k in inc_keys}
).T.round(6)

print("=" * 60)
print("  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics")
print("=" * 60)
display(df_incs.style.format("{:.4f}"))

  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics


,Reference,Generated
mean,0.0000,0.0000
std,0.0296,0.0263
min,-1.3863,-0.8447
q01,-0.0805,-0.0652
q99,0.0820,0.0660
max,1.0291,0.8437
skewness,0.2566,0.1372
excess_kurtosis,26.8585,8.7306


## 2b. Aggregate statistics — full stock series vs generated

In [203]:
# Load full stock series from REF_DIRECTORY (one per stock, no windowing)
ref_full_paths = []
for fp in sorted(Path(REF_DIRECTORY).glob("*.csv")):
    series = pd.read_csv(fp)["log_adj_close"].values.astype(np.float64)
    ref_full_paths.append(series)

print(f"Full stock series loaded : {len(ref_full_paths)}")
print(f"Lengths — min={min(len(s) for s in ref_full_paths)}  "
      f"max={max(len(s) for s in ref_full_paths)}  "
      f"mean={np.mean([len(s) for s in ref_full_paths]):.0f}")

ref_full_stats = compute_stats(ref_full_paths, "Reference (full series)")

# ── Levels table ──────────────────────────────────────────────────────────────
level_keys = ["n_paths", "n_values", "mean", "std", "min", "q01", "q05",
              "q25", "q50", "q75", "q95", "q99", "max", "skewness", "excess_kurtosis"]
df_levels_full = pd.DataFrame(
    {k: {"Reference (full)": ref_full_stats[k], "Generated": gen_stats[k]} for k in level_keys}
).T.round(4)

print("\n" + "=" * 60)
print("  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)")
print("=" * 60)
display(df_levels_full.style.format("{:.4f}"))

# ── Increments table ──────────────────────────────────────────────────────────
inc_keys = ["inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99",
            "inc_max", "inc_skewness", "inc_excess_kurt"]
rename = {
    "inc_mean": "mean", "inc_std": "std", "inc_min": "min",
    "inc_q01": "q01", "inc_q99": "q99", "inc_max": "max",
    "inc_skewness": "skewness", "inc_excess_kurt": "excess_kurtosis",
}
df_incs_full = pd.DataFrame(
    {rename[k]: {"Reference (full)": ref_full_stats[k], "Generated": gen_stats[k]}
     for k in inc_keys}
).T.round(6)

print("\n" + "=" * 60)
print("  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics")
print("=" * 60)
display(df_incs_full.style.format("{:.4f}"))

Full stock series loaded : 210
Lengths — min=10083  max=16175  mean=12493

  LOG_ADJ_CLOSE  —  Aggregate statistics (levels)


,Reference (full),Generated
n_paths,210.0000,1024.0000
n_values,2623490.0000,2097152.0000
mean,0.0005,0.0004
std,0.0209,0.0186
min,-0.9363,-0.8925
q01,-0.0564,-0.0463
q05,-0.0292,-0.0283
q25,-0.0086,-0.0105
q50,0.0000,0.0004
q75,0.0095,0.0111



  INCREMENTS  ΔX_t = X_t − X_{t−1}  —  Aggregate statistics


,Reference (full),Generated
mean,-0.0000,0.0000
std,0.0297,0.0263
min,-1.3863,-0.8447
q01,-0.0805,-0.0652
q99,0.0820,0.0660
max,1.0291,0.8437
skewness,0.2227,0.1372
excess_kurtosis,29.4105,8.7306


## Per-window statistics

In [204]:
def per_window_stats(paths: list[np.ndarray]) -> pd.DataFrame:
    rows = []
    for p in paths:
        incs = np.diff(p)
        rows.append({
            "mean"         : float(p.mean()),
            "std"          : float(p.std()),
            "skewness"     : float(scipy_stats.skew(p)),
            "excess_kurt"  : float(scipy_stats.kurtosis(p, fisher=True)),
            "zero_ratio"   : float(np.mean(p == 0.0)),
            "inc_std"      : float(incs.std()),
            "inc_skewness" : float(scipy_stats.skew(incs)),
            "inc_excess_kurt": float(scipy_stats.kurtosis(incs, fisher=True)),
        })
    return pd.DataFrame(rows)

ref_pw = per_window_stats(ref_paths)
gen_pw = per_window_stats(gen_paths)

# ── Summary table: mean ± std of each per-window stat ────────────────────────
summary = pd.DataFrame({
    "Reference  mean": ref_pw.mean(),
    "Reference  std" : ref_pw.std(),
    "Generated  mean": gen_pw.mean(),
    "Generated  std" : gen_pw.std(),
}).round(5)
print("Per-window statistics — summary (mean ± std across windows)")
display(summary.style.format("{:.4f}"))

# ── Violin plots ──────────────────────────────────────────────────────────────
metrics = list(ref_pw.columns)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))

for ax, metric in zip(axes.flat, metrics):
    data  = [ref_pw[metric].values, gen_pw[metric].values]
    parts = ax.violinplot(data, positions=[0, 1], showmedians=True, showextrema=True)
    parts["bodies"][0].set_facecolor("tab:blue");   parts["bodies"][0].set_alpha(0.55)
    parts["bodies"][1].set_facecolor("tab:orange"); parts["bodies"][1].set_alpha(0.55)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Reference", "Generated"])
    ax.set_title(metric)
    ax.grid(axis="y", linewidth=0.4)

plt.suptitle("Distribution of per-window statistics", y=1.02)
plt.tight_layout()
plt.show()


Per-window statistics — summary (mean ± std across windows)


,Reference mean,Reference std,Generated mean,Generated std
mean,0.0005,0.0004,0.0003,0.0005
std,0.0196,0.0069,0.0183,0.0031
skewness,-0.3403,1.3745,-0.1347,0.8649
excess_kurt,14.0761,36.3552,5.8164,22.1054
zero_ratio,0.0717,0.0936,0.0000,0.0000
inc_std,0.0279,0.0100,0.0259,0.0044
inc_skewness,0.2385,0.4993,0.1156,0.2498
inc_excess_kurt,11.2795,23.0052,4.4053,11.5444


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\2342577457.py:45: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Kurtosis offenders

The top offenders are identified through computation of the increments so as to identify most leptokurtic distribution, which have very fat tails and sharp peaks.

In [205]:
# ── Kurtosis diagnosis: worst windows by increment excess kurtosis ─────────────
TOP_K      = 10   # number of worst windows to inspect
TOP_N_INCS = 5    # largest increments to report per window

def print_worst_kurtosis_windows(pw: pd.DataFrame, paths: list, label: str,
                                  top_k: int, top_n: int) -> None:
    worst = pw.nlargest(top_k, "inc_excess_kurt")
    print(f"{'='*70}")
    print(f"Top-{top_k} {label} windows by increment excess kurtosis")
    print(f"{'='*70}\n")
    for rank, (win_idx, row) in enumerate(worst.iterrows(), start=1):
        path     = paths[win_idx]
        incs     = np.diff(path)
        abs_incs = np.abs(incs)
        print(f"Rank {rank:2d} | window index {win_idx}")
        print(f"  {'mean':<20s}: {row['mean']:+.6f}")
        print(f"  {'std':<20s}: {row['std']:.6f}")
        print(f"  {'skewness':<20s}: {row['skewness']:+.4f}")
        print(f"  {'excess_kurt (levels)':<20s}: {row['excess_kurt']:+.4f}")
        print(f"  {'inc_std':<20s}: {row['inc_std']:.6f}")
        print(f"  {'inc_skewness':<20s}: {row['inc_skewness']:+.4f}")
        print(f"  {'inc_excess_kurt':<20s}: {row['inc_excess_kurt']:+.4f}")
        top_pos = np.argsort(abs_incs)[::-1][:top_n]
        print(f"\n  Top-{top_n} largest |Δx_t|  (position l in [0, {len(incs)-1}]):")
        print(f"  {'l':>6}  {'Δx':>12}  {'|Δx|':>10}  {'x[l]':>12}  {'x[l+1]':>12}")
        for pos in top_pos:
            print(f"  {pos:>6d}  {incs[pos]:>+12.6f}  {abs_incs[pos]:>10.6f}"
                  f"  {path[pos]:>+12.6f}  {path[pos+1]:>+12.6f}")
        print()

print_worst_kurtosis_windows(gen_pw, gen_paths, "Generated", TOP_K, TOP_N_INCS)
print_worst_kurtosis_windows(ref_pw, ref_paths, "Reference", TOP_K, TOP_N_INCS)

Top-10 Generated windows by increment excess kurtosis

Rank  1 | window index 41
  mean                : +0.000369
  std                 : 0.022350
  skewness            : -14.3365
  excess_kurt (levels): +440.7600
  inc_std             : 0.031071
  inc_skewness        : +3.2330
  inc_excess_kurt     : +224.5946

  Top-5 largest |Δx_t|  (position l in [0, 2046]):
       l            Δx        |Δx|          x[l]        x[l+1]
    2045     +0.743648    0.743648     -0.689183     +0.054465
    2044     -0.595963    0.595963     -0.093221     -0.689183
    1893     -0.119251    0.119251     +0.030032     -0.089219
     905     +0.103520    0.103520     -0.019333     +0.084188
    1007     -0.092975    0.092975     +0.045854     -0.047121

Rank  2 | window index 416
  mean                : +0.000416
  std                 : 0.027627
  skewness            : -10.8743
  excess_kurt (levels): +301.5631
  inc_std             : 0.041322
  inc_skewness        : -0.1024
  inc_excess_kurt     : +156.

In [206]:
# if TRAINING_DATASET_SIZE > 15 or TRAINING_DATASET_SIZE == 1.0:
# ── Aggregate statistics after removing the N_DROP most leptokurtic windows ───
N_DROP = 15   # number of worst-kurtosis windows to exclude from each dataset

gen_drop_idx = set(gen_pw.nlargest(N_DROP, "inc_excess_kurt").index)
ref_drop_idx = set(ref_pw.nlargest(N_DROP, "inc_excess_kurt").index)

gen_paths_trim = [p for i, p in enumerate(gen_paths) if i not in gen_drop_idx]
ref_paths_trim = [p for i, p in enumerate(ref_paths) if i not in ref_drop_idx]

print(f"Generated : {len(gen_paths)} → {len(gen_paths_trim)} windows "
      f"(removed {len(gen_drop_idx)} most leptokurtic)")
print(f"Reference : {len(ref_paths)} → {len(ref_paths_trim)} windows "
      f"(removed {len(ref_drop_idx)} most leptokurtic)\n")

ref_stats_trim = compute_stats(ref_paths_trim, "Reference (trimmed)")
gen_stats_trim = compute_stats(gen_paths_trim, "Generated (trimmed)")

keys = ["mean", "std", "min", "q01", "q99", "max", "skewness", "excess_kurtosis",
      "inc_mean", "inc_std", "inc_min", "inc_q01", "inc_q99", "inc_max",
      "inc_skewness", "inc_excess_kurt"]

df_cmp = pd.DataFrame({
"Gen  full"   : {k: gen_stats[k]      for k in keys},
"Gen  trimmed": {k: gen_stats_trim[k] for k in keys},
"Ref  full"   : {k: ref_stats[k]      for k in keys},
"Ref  trimmed": {k: ref_stats_trim[k] for k in keys},
}).round(6)

print(f"Aggregate statistics — full vs trimmed (top-{N_DROP} leptokurtic windows removed)")
display(df_cmp.style.format("{:.6f}"))

Generated : 1024 → 1009 windows (removed 15 most leptokurtic)
Reference : 5290 → 5275 windows (removed 15 most leptokurtic)

Aggregate statistics — full vs trimmed (top-15 leptokurtic windows removed)


,Gen full,Gen trimmed,Ref full,Ref trimmed
mean,0.000351,0.000352,0.000459,0.000460
std,0.018582,0.018510,0.020802,0.020798
min,-0.892527,-0.389336,-0.936258,-0.936258
q01,-0.046339,-0.046283,-0.056155,-0.056193
q99,0.047318,0.047267,0.058261,0.058269
max,0.604577,0.302700,0.693147,0.693147
skewness,-0.228543,-0.113368,-0.538746,-0.487746
excess_kurtosis,13.178194,5.703987,36.063195,34.281046
inc_mean,0.000000,0.000000,0.000000,0.000000
inc_std,0.026276,0.026170,0.029619,0.029615


In [207]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (pw, paths, label, color) in zip(axes, [
    (ref_pw, ref_paths, "Reference", "tab:blue"),
    (gen_pw, gen_paths, "Generated", "tab:orange"),
]):
    worst_idx = pw["inc_excess_kurt"].idxmax()
    p = paths[worst_idx]
    ax.plot(p, linewidth=0.8, color=color)
    ax.set_title(f"{label} — most leptokurtic window\n"
                 f"inc_excess_kurt={pw.loc[worst_idx, 'inc_excess_kurt']:.2f}  "
                 f"idx={worst_idx}")
    ax.set_xlabel("step t")
    ax.set_ylabel("log_adj_close")
    ax.grid(True, linewidth=0.3)

plt.tight_layout()
plt.show()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\1686780548.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [208]:
# ── Compare N individual windows with custom range (user-specified) ──────────
N_WINDOWS = 1     # ← change this to select how many paths to visualize
START_IDX = 50       # ← start index (0 = beginning)
END_IDX   = 150   # ← end index (None = full length, or e.g. 500, 1000, etc.)

n_ref = min(N_WINDOWS, len(ref_paths))
n_gen = min(N_WINDOWS, len(gen_paths))

ref_idx = rng.choice(len(ref_paths), size=n_ref, replace=False)
gen_idx = rng.choice(len(gen_paths), size=n_gen, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Reference paths ─────────────────────────────────────────────────────────
ax = axes[0]
for i in ref_idx:
    p = ref_paths[i][START_IDX:END_IDX]
    ax.plot(np.arange(START_IDX, START_IDX + len(p)), p, 
            alpha=0.4, linewidth=0.9, color="tab:blue")
ax.set_title(f"Reference (N={n_ref}, range [{START_IDX}:{END_IDX}])", 
             fontsize=12, fontweight="bold")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close")
ax.grid(True, linewidth=0.3)

# ── Generated paths ─────────────────────────────────────────────────────────
ax = axes[1]
for i in gen_idx:
    p = gen_paths[i][START_IDX:END_IDX]
    ax.plot(np.arange(START_IDX, START_IDX + len(p)), p, 
            alpha=0.4, linewidth=0.9, color="tab:orange")
ax.set_title(f"Generated (N={n_gen}, range [{START_IDX}:{END_IDX}])", 
             fontsize=12, fontweight="bold")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close")
ax.grid(True, linewidth=0.3)

plt.suptitle(f"Window comparison (N={N_WINDOWS}, range [{START_IDX}:{END_IDX}])", 
             y=1.00, fontsize=13)
plt.tight_layout()
plt.show()

print(f"Plotted {n_ref} reference and {n_gen} generated paths")
print(f"Range: [{START_IDX}:{END_IDX}]")


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\3126294916.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Plotted 1 reference and 1 generated paths
Range: [50:150]


## 3. Level distribution: histogram, QQ plot, ECDF

In [209]:
gen_checkpoint_name = os.path.basename(GEN_DIRECTORY.rstrip('/'))
ref_checkpoint_name = os.path.basename(REF_DIRECTORY.rstrip('/'))

In [210]:
ref_flat = np.concatenate(ref_paths)
gen_flat = np.concatenate(gen_paths)

ks_stat_lev, ks_p_lev = scipy_stats.ks_2samp(ref_flat, gen_flat)
print(f"Levels KS statistic = {ks_stat_lev:.4f}  |  p-value = {ks_p_lev:.4e}")

combined = np.concatenate([ref_flat, gen_flat])
lo, hi   = np.quantile(combined, [0.002, 0.998])
bins     = np.linspace(lo, hi, 80)
xs       = np.linspace(lo, hi, 400)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Histogram / density
ax = axes[0]
ax.hist(ref_flat, bins=bins, density=True, alpha=0.45, label="Reference", color="tab:blue")
ax.hist(gen_flat, bins=bins, density=True, alpha=0.45, label="Generated", color="tab:orange")
ax.set_title("Level marginal distribution")
ax.set_xlabel("log_adj_close")
ax.set_ylabel("density")
ax.legend(fontsize=8)

# (b) Empirical QQ plot (ref quantiles vs gen quantiles)
ax = axes[1]
n_q  = min(len(ref_flat), len(gen_flat), 5_000)
probs = np.linspace(0.01, 0.99, n_q)
q_ref = np.quantile(ref_flat, probs)
q_gen = np.quantile(gen_flat, probs)
ax.scatter(q_ref, q_gen, s=3, alpha=0.4, color="steelblue")
lims = [min(q_ref.min(), q_gen.min()), max(q_ref.max(), q_gen.max())]
ax.plot(lims, lims, "r--", linewidth=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — levels\n(ref quantiles vs gen quantiles)")
ax.set_xlabel("Reference quantiles")
ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

# (c) ECDF
ax = axes[2]
for vals, label, color in [
    (ref_flat, "Reference", "tab:blue"),
    (gen_flat, "Generated", "tab:orange")
]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, linewidth=1.2)
ax.set_xlim(lo, hi)
ax.set_title(f"ECDF — levels  (KS={ks_stat_lev:.3f}, p={ks_p_lev:.2e})")
ax.set_xlabel("log_adj_close")
ax.set_ylabel("cumulative probability")
ax.legend(fontsize=8)

plt.suptitle("log_adj_close — level distributions", y=1.02)
plt.tight_layout()
plt.savefig(f"../images/comparison/level_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

Levels KS statistic = 0.0389  |  p-value = 0.0000e+00


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\31005602.py:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\31005602.py:53: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(f"../images/comparison/level_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\31005602.py:54: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Increment distribution: histogram, QQ plot, ECDF

In [211]:
ref_incs = np.concatenate([np.diff(p) for p in ref_paths if len(p) > 1])
gen_incs = np.concatenate([np.diff(p) for p in gen_paths if len(p) > 1])

ks_stat_inc, ks_p_inc = scipy_stats.ks_2samp(ref_incs, gen_incs)
print(f"Increments KS statistic = {ks_stat_inc:.4f}  |  p-value = {ks_p_inc:.4e}")

combined_inc = np.concatenate([ref_incs, gen_incs])
lo_i, hi_i  = np.quantile(combined_inc, [0.002, 0.998])
bins_i      = np.linspace(lo_i, hi_i, 80)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# (a) Histogram / density
ax = axes[0]
ax.hist(ref_incs, bins=bins_i, density=True, alpha=0.45, label="Reference", color="tab:blue")
ax.hist(gen_incs, bins=bins_i, density=True, alpha=0.45, label="Generated", color="tab:orange")
ax.set_title("Increment marginal distribution")
ax.set_xlabel("ΔX = log_adj_close[t] − log_adj_close[t−1]")
ax.set_ylabel("density")
ax.legend(fontsize=8)

# (b) Empirical QQ plot
ax = axes[1]
n_q  = min(len(ref_incs), len(gen_incs), 5_000)
probs = np.linspace(0.01, 0.99, n_q)
q_ref_i = np.quantile(ref_incs, probs)
q_gen_i = np.quantile(gen_incs, probs)
ax.scatter(q_ref_i, q_gen_i, s=3, alpha=0.4, color="steelblue")
lims_i = [min(q_ref_i.min(), q_gen_i.min()), max(q_ref_i.max(), q_gen_i.max())]
ax.plot(lims_i, lims_i, "r--", linewidth=1.2, label="y = x (perfect)")
ax.set_title("QQ plot — increments\n(ref quantiles vs gen quantiles)")
ax.set_xlabel("Reference quantiles")
ax.set_ylabel("Generated quantiles")
ax.legend(fontsize=8)

# (c) ECDF
ax = axes[2]
for vals, label, color in [
    (ref_incs, "Reference", "tab:blue"),
    (gen_incs, "Generated", "tab:orange")
]:
    s = np.sort(vals)
    ax.plot(s, np.arange(1, len(s) + 1) / len(s), label=label, linewidth=1.2)
ax.set_xlim(lo_i, hi_i)
ax.set_title(f"ECDF — increments  (KS={ks_stat_inc:.3f}, p={ks_p_inc:.2e})")
ax.set_xlabel("ΔX")
ax.set_ylabel("cumulative probability")
ax.legend(fontsize=8)

plt.suptitle("Increments ΔX_t — distributions", y=1.02)
plt.tight_layout()
plt.savefig(f"../images/comparison/increments_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

Increments KS statistic = 0.0318  |  p-value = 0.0000e+00


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\895376522.py:51: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.tight_layout()
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\895376522.py:52: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  plt.savefig(f"../images/comparison/increments_distributions_comparison_{gen_checkpoint_name}.png", dpi=300)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\895376522.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Sample paths

Visual check: do the shapes, scale, and fan-out look alike?

Reference paths may be longer; if so, a random window of the generated-path length is
extracted (using `SEED`) for a fair apples-to-apples visual.

In [212]:
N_SHOW   = 20
GEN_LEN  = int(np.median(gen_lens))   # typical generated path length

# Sample random paths from each dataset
ref_idx = rng.choice(len(ref_paths), size=min(N_SHOW, len(ref_paths)), replace=False)
gen_idx = rng.choice(len(gen_paths), size=min(N_SHOW, len(gen_paths)), replace=False)

def extract_window(path: np.ndarray, window_len: int, rng: np.random.Generator) -> np.ndarray:
    """Extract a random window of `window_len` from `path`."""
    if len(path) <= window_len:
        return path
    start = rng.integers(0, len(path) - window_len + 1)
    return path[start : start + window_len]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

# Reference
ax = axes[0]
for i in ref_idx:
    p = extract_window(ref_paths[i], GEN_LEN, rng)
    p = p - p[0]   # anchor at 0
    ax.plot(np.arange(len(p)), p, alpha=0.35, linewidth=0.8, color="tab:blue")
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"Reference paths (n={len(ref_idx)}, window={GEN_LEN} steps, anchored at 0)")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close − log_adj_close[0]")

# Generated
ax = axes[1]
for i in gen_idx:
    p = gen_paths[i]
    p = p - p[0]   # anchor at 0
    ax.plot(np.arange(len(p)), p, alpha=0.35, linewidth=0.8, color="tab:orange")
ax.axhline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_title(f"Generated paths (n={len(gen_idx)}, len={GEN_LEN} steps, anchored at 0)")
ax.set_xlabel("step t")
ax.set_ylabel("log_adj_close − log_adj_close[0]")

plt.suptitle("Sample paths comparison (all anchored at 0)", y=1.02)
plt.tight_layout()
plt.savefig
plt.savefig(f"../images/comparison/sample_paths_comparison_{gen_checkpoint_name}.png", dpi=300)
plt.show()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12184\2721214629.py:43: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Autocorrelation: levels vs increments

**Rationale**:
- *Levels*: log-price paths should show high persistence (ACF ≈ 1, slow decay).
- *Increments*: log-returns should be nearly i.i.d. (ACF ≈ 0 at all lags ≥ 1).

We average ACFs across multiple paths for a stable estimate.  
For reference paths longer than `GEN_LEN` we extract the first `GEN_LEN` steps.

In [213]:
# N_ACF_PATHS = min(50, len(ref_paths), len(gen_paths))
# LAGS        = min(30, GEN_LEN // 2)

# def mean_acf_across_paths(
#     paths: list[np.ndarray],
#     n_paths: int,
#     lags: int,
#     use_diff: bool = False,
#     max_len: int | None = None,
#     rng: np.random.Generator | None = None,
# ) -> np.ndarray:
#     """Average ACF (lags 1..lags) across up to `n_paths` paths."""
#     indices = (
#         rng.choice(len(paths), size=n_paths, replace=False)
#         if rng is not None
#         else np.arange(min(n_paths, len(paths)))
#     )
#     acfs = []
#     for i in indices:
#         x = paths[i]
#         if max_len is not None and len(x) > max_len:
#             x = x[:max_len]   # align to generated path length
#         if use_diff:
#             x = np.diff(x)
#         if len(x) <= lags:
#             continue
#         acfs.append(sm_acf(x, nlags=lags, fft=True)[1:])   # skip lag-0 (=1)
#     return np.mean(acfs, axis=0) if acfs else np.zeros(lags)

# lag_axis = np.arange(1, LAGS + 1)

# ref_acf_lev = mean_acf_across_paths(ref_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=False, max_len=GEN_LEN, rng=rng)
# gen_acf_lev = mean_acf_across_paths(gen_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=False, max_len=None, rng=rng)
# ref_acf_inc = mean_acf_across_paths(ref_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=True,  max_len=GEN_LEN, rng=rng)
# gen_acf_inc = mean_acf_across_paths(gen_paths, N_ACF_PATHS, LAGS,
#                                     use_diff=True,  max_len=None, rng=rng)

# fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ax = axes[0]
# ax.plot(lag_axis, ref_acf_lev, label="Reference", color="tab:blue",   linewidth=1.4)
# ax.plot(lag_axis, gen_acf_lev, label="Generated", color="tab:orange", linewidth=1.4)
# ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
# ax.set_title("ACF of levels  (should be ≈ 1, slow decay)")
# ax.set_xlabel("lag")
# ax.set_ylabel("autocorrelation")
# ax.set_ylim(-0.2, 1.05)
# ax.legend(fontsize=9)

# ax = axes[1]
# ax.plot(lag_axis, ref_acf_inc, label="Reference", color="tab:blue",   linewidth=1.4)
# ax.plot(lag_axis, gen_acf_inc, label="Generated", color="tab:orange", linewidth=1.4)
# ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
# ax.set_title("ACF of increments  (should be ≈ 0 at all lags)")
# ax.set_xlabel("lag")
# ax.set_ylabel("autocorrelation")
# ax.set_ylim(-0.3, 1.05)
# ax.legend(fontsize=9)

# plt.suptitle(f"Autocorrelation structure  (avg over {N_ACF_PATHS} paths)", y=1.02)
# plt.tight_layout()
# plt.show()

# print(f"Ref  level  ACF at lag-1: {ref_acf_lev[0]:.3f}  |  "
#       f"Gen  level  ACF at lag-1: {gen_acf_lev[0]:.3f}")
# print(f"Ref  inc    ACF at lag-1: {ref_acf_inc[0]:.3f}  |  "
#       f"Gen  inc    ACF at lag-1: {gen_acf_inc[0]:.3f}")

# 8. Paper's FTS evaluations metrics
These metrics are written to work on **LOG-RETURN**.

In [214]:
sys.path.append(os.path.abspath(".."))

import replication.stylized_facts as sf
import numpy as np

In [215]:
# Reference: full stock series (not windows)
ref_full = {}
for csv_path in sorted(Path(REF_DIRECTORY).glob("*.csv")):
    df = pd.read_csv(csv_path)
    ref_full[csv_path.stem] = df["log_adj_close"].to_numpy(dtype=float)

ref_full_obj    = np.empty(len(ref_full), dtype=object)
ref_full_pooled = np.concatenate(list(ref_full.values()))
for i, r in enumerate(ref_full.values()):
    ref_full_obj[i] = r

# Generated: already 2048-pt independent samples — use as-is
# gen_paths_obj and gen_paths_arr unchanged


In [216]:
# Use sf.distribution() with normalize=True and scale='log'
output_dir = Path("../images/stylized_facts_output_heavy_tails")
output_dir.mkdir(exist_ok=True, parents=True)


In [217]:
print(gen_paths[1][:5], "\n", gen_paths[0][-5:])

[ 0.00410387 -0.01332373 -0.01050575 -0.0093191  -0.02682452] 
 [ 0.01077308  0.00040339  0.01171346  0.01455438 -0.01350568]


In [218]:
# Prepare data as object arrays (one array per path)
# ref_paths_arr = np.array(ref_paths, dtype=object)
# gen_paths_arr = np.array(gen_paths, dtype=object)

# Using np.concatenate
# ref_paths_arr = np.concatenate(ref_paths)
gen_paths_arr = np.concatenate(gen_paths)

# If ref_paths and gen_paths are lists:
# ref_paths_obj = np.empty(len(ref_paths), dtype=object)
# for i, r in enumerate(ref_paths):
#     ref_paths_obj[i] = r

gen_paths_obj = np.empty(len(gen_paths), dtype=object)
for i, g in enumerate(gen_paths):
    gen_paths_obj[i] = g

# Extract checkpoint names (last part after "/")
gen_checkpoint_name = os.path.basename(GEN_DIRECTORY.rstrip('/'))
ref_checkpoint_name = os.path.basename(REF_DIRECTORY.rstrip('/'))

In [219]:
# # print(ref_paths[:5], "\n", ref_paths.shape)
# print(ref_paths_arr[:5],"\n", ref_paths_arr.shape)
print(gen_paths_obj[:5],"\n", gen_paths_obj.shape)

[array([-0.01828594, -0.01875875,  0.01205785, ...,  0.01171346,
         0.01455438, -0.01350568])
 array([ 0.00410387, -0.01332373, -0.01050575, ..., -0.00110717,
        -0.00155877, -0.00372984])
 array([0.01235233, 0.00204138, 0.00336272, ..., 0.00352718, 0.01096956,
        0.01182286])
 array([-0.02552452, -0.04121103,  0.00631615, ...,  0.00686869,
         0.00125291, -0.00116877])
 array([-0.0168276 ,  0.01565296,  0.03683314, ..., -0.01929787,
         0.00219664,  0.00715646])                              ] 
 (1024,)


In [220]:
sf.distribution(
    ref_full_pooled,
    file_name=str(output_dir / f"generated_distribution_{ref_checkpoint_name}"),
    scale="log",
    multiple=False,
    normalize=True,
    granuality=100,
)

In [221]:
sf.distribution(
    gen_paths_arr,
    file_name=str(output_dir / f"generated_distribution_{gen_checkpoint_name}"),
    scale="log",
    multiple=False,
    normalize=True,
    granuality=100,
)

In [222]:
# sf.acf(
#     ref_full_obj,
#     file_name=str(output_dir / f"reference_volatility_clustering_{ref_checkpoint_name}"),
#     for_abs=True,
#     multiple=True,
#     fit=False,
#     scale="log",
#     max_lag=1000,
# )


In [223]:
# sf.acf(
#     gen_paths_obj,
#     file_name=str(output_dir / f"generated_volatility_clustering__{gen_checkpoint_name}"),
#     for_abs=True,
#     multiple=True,
#     fit=False,
#     scale="log",
#     max_lag=1000,
# )

In [224]:
ref_lev = sf.leverage_effect(
    ref_full_obj,
    file_name=str(output_dir / f"reference_leverage_effect_{ref_checkpoint_name}"),
    multiple=True,
    min_lag=1,
    max_lag=100,
)

gen_lev = sf.leverage_effect(
    gen_paths_obj,
    file_name=str(output_dir / f"generated_leverage_effect_{gen_checkpoint_name}"),
    multiple=True,
    min_lag=1,
    max_lag=100,
)


In [233]:
import powerlaw

def fit_powerlaw(returns, max_sample=15000, seed=42):
    x = np.abs(np.asarray(returns, dtype=float))
    x = x[np.isfinite(x) & (x > 0)]

    if len(x) > max_sample:
        rng_local = np.random.default_rng(seed)
        x = rng_local.choice(x, size=max_sample, replace=False)

    f = powerlaw.Fit(x, discrete=False, verbose=True,
                     parameter_ranges={"alpha": [1.5, 10.0]})

    x_tail = x[x >= f.power_law.xmin]
    alpha_mle = 1 + len(x_tail) / np.sum(np.log(x_tail / f.power_law.xmin))

    return {
        "alpha"    : f.power_law.alpha,
        "alpha_mle": alpha_mle,
        "xmin"     : f.power_law.xmin,
        "ks"       : f.power_law.D,
        "n_total"  : len(x),
        "n_tail"   : len(x_tail),
    }


for paths, label in [(list(ref_full.values()), "Reference"), (gen_paths, "Generated")]:
    incs = np.concatenate(list(paths))
    r = fit_powerlaw(incs)
    print(f"{label}")
    print(f"  alpha (powerlaw) : {r['alpha']:.4f}")
    print(f"  alpha (MLE)      : {r['alpha_mle']:.4f}")
    print(f"  xmin             : {r['xmin']:.6f}")
    print(f"  KS distance      : {r['ks']:.4f}")
    print(f"  n_total          : {r['n_total']:,}")
    print(f"  n_tail           : {r['n_tail']:,}")
    print()

Calculating best minimal value for power law fit


Fitting xmin: 100%|██████████| 14974/14974 [00:30<00:00, 490.20it/s] 


Reference
  alpha (powerlaw) : 3.8245
  alpha (MLE)      : 3.8245
  xmin             : 0.041797
  KS distance      : 0.0174
  n_total          : 15,000
  n_tail           : 825

Calculating best minimal value for power law fit


Fitting xmin: 100%|██████████| 14993/14993 [00:45<00:00, 328.20it/s]

Generated
  alpha (powerlaw) : 4.4146
  alpha (MLE)      : 4.4146
  xmin             : 0.035152
  KS distance      : 0.0152
  n_total          : 15,000
  n_tail           : 884

